# PMLL-memory-MCP — Kaggle Dataset Installer (Online)

**Purpose:** Download `pmll-memory-mcp` and all its dependencies into a local directory,
then save that directory as output.  Once this notebook is committed, Kaggle will
make the output available as a **Dataset** that can be attached to any *offline* notebook
as an input — allowing `pip install --no-index --find-links` to work without internet.

## Instructions
1. Open this notebook in Kaggle (`+ New Notebook → Import from GitHub`).
2. **Settings → Internet → On** (internet must be ON for this installer notebook).
3. Click **Run All**.
4. After the run completes, click **Save Version** → choose **Save & Run All** → tick
   **"Make output available as a dataset"** and name the dataset `pmll-memory-mcp-wheels`.
5. Attach that dataset to the offline notebook
   (`arc_agi3_worldmodel_offline_kaggle.ipynb`) as an input.


## 1 — Download `pmll-memory-mcp` wheels to a local directory

In [ ]:
import subprocess, sys
from pathlib import Path

WHEELS_DIR = Path("/kaggle/working/pmll_wheels")
WHEELS_DIR.mkdir(parents=True, exist_ok=True)

# Download pmll-memory-mcp plus every dependency into WHEELS_DIR
subprocess.check_call([
    sys.executable, "-m", "pip", "download",
    "pmll-memory-mcp",
    "--dest", str(WHEELS_DIR),
    "--no-cache-dir",
])

wheels = list(WHEELS_DIR.glob("*.whl")) + list(WHEELS_DIR.glob("*.tar.gz"))
print(f"Downloaded {len(wheels)} package file(s) to {WHEELS_DIR}:")
for w in sorted(wheels):
    print(f"  {w.name}")

## 2 — Smoke-test: install from the local directory and import

In [ ]:
# Install from the local wheels directory to verify they are complete
subprocess.check_call([
    sys.executable, "-m", "pip", "install",
    "pmll-memory-mcp",
    "--no-index",
    "--find-links", str(WHEELS_DIR),
    "--quiet",
])

# Basic import sanity check
import importlib.util
spec = importlib.util.find_spec("pmll_memory_mcp")
if spec is not None:
    print(f"pmll_memory_mcp found at: {spec.origin}")
else:
    print("pmll_memory_mcp module not directly importable (MCP-server package — OK).")
    print("Verifying CLI entry point instead:")
    result = subprocess.run(
        [sys.executable, "-m", "pip", "show", "pmll-memory-mcp"],
        capture_output=True, text=True,
    )
    print(result.stdout)

print("Smoke-test passed — wheels are installable offline.")

## 3 — List output for Kaggle dataset creation

Everything written to `/kaggle/working/` is included in the dataset output.
The offline notebook will find the wheels at `/kaggle/input/pmll-memory-mcp-wheels/pmll_wheels/`.

In [ ]:
import os
total_bytes = 0
print("Contents of /kaggle/working/pmll_wheels/:")
for f in sorted(WHEELS_DIR.iterdir()):
    size = f.stat().st_size
    total_bytes += size
    print(f"  {f.name:60s}  {size / 1024:.1f} KB")
print(f"\nTotal: {total_bytes / 1024 / 1024:.2f} MB")
print("\nDataset will be available at:")
print("  /kaggle/input/pmll-memory-mcp-wheels/pmll_wheels/*.whl")